# RBT12 Livre — Simulações, Histórico e Edição Manual por Anexo

Este notebook mostra como **desacoplar o RBT12** do cálculo automático, permitindo:

1. **Simular cenários futuros** (otimista / base / pessimista) para o RBT12.
2. **Visualizar o histórico** de RBT12 por anexo (automático, simulado e editável).
3. **Digitar o RBT12 manualmente** dentro de cada anexo, respeitando limites.

> **Contexto:** no app (`simples_app`), o RBT12 hoje é sempre derivado dos 12 meses de faturamento
> (janela relativa à data atual). Isso torna extratos antigos invisíveis e impede simulações.
> Aqui exploramos o modelo de dados que habilita a edição livre.

**Estrutura:**
1. Carregar Dataset e Bibliotecas
2. Inspecionar Estrutura de Anexos e RBT12
3. Desacoplar RBT12 do Cálculo Automático
4. Simular RBT12 Futuro
5. Visualizar Histórico de RBT12 por Anexo
6. Entrada Manual de RBT12 por Anexo
7. Validar Consistência dos Cenários

## 1. Carregar Dataset e Bibliotecas

Importamos `pandas`, `numpy` e `matplotlib`, e carregamos a base com as colunas
`anexo`, `competencia` e `rbt12`. Garantimos que `competencia` esteja em `datetime`.
Como a base real pode não estar disponível, geramos um **dataset de demonstração**
que reproduz o formato usado pelo app (12 meses por anexo + meses históricos).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("pandas", pd.__version__, "| numpy", np.__version__)

ANO_HOJE = 2026
MES_HOJE = 9  # setembro/2026 (data atual do sistema)


def _gerar_base_demo(anexos=("I", "III"), meses=72, seed=42):
    """Cria uma base no formato do app: um RBT12 por anexo/competência.

    Reproduz o cenário descrito: meses históricos (ex.: 2021) que existem no
    banco mas ficam fora da janela visível de 12 meses.
    """
    rng = np.random.default_rng(seed)
    competencias = pd.date_range(
        end=pd.Timestamp(ANO_HOJE, MES_HOJE, 1), periods=meses, freq="MS"
    )
    linhas = []
    for anexo in anexos:
        faturamento = rng.normal(40_000, 6_000, size=meses).clip(min=5_000)
        # RBT12 = média dos 12 meses * 12 (mesma fórmula do app/planilha C15).
        for i, comp in enumerate(competencias):
            janela = faturamento[max(0, i - 11): i + 1]
            rbt12 = 12 * janela.sum() / len(janela)
            linhas.append(
                {
                    "anexo": anexo,
                    "competencia": comp,
                    "faturamento_mes": round(float(faturamento[i]), 2),
                    "rbt12": round(float(rbt12), 2),
                }
            )
    return pd.DataFrame(linhas)


df = _gerar_base_demo()

# Garantia de tipo: competência sempre em datetime.
df["competencia"] = pd.to_datetime(df["competencia"])
df = df.sort_values(["anexo", "competencia"]).reset_index(drop=True)

print(f"Linhas: {len(df)} | Anexos: {sorted(df['anexo'].unique())}")
df.head(8)

## 2. Inspecionar Estrutura de Anexos e RBT12

Antes de mudar o comportamento, entendemos o estado atual: quais anexos existem,
tipos das colunas, valores nulos e o **range** de RBT12. O gráfico de distribuição
por anexo dá a fotografia inicial.

In [ ]:
print("=== Tipos ===")
print(df.dtypes)

print("\n=== Valores nulos ===")
print(df.isna().sum())

print("\n=== Anexos únicos ===")
print(sorted(df["anexo"].unique()))

print("\n=== Range de competência ===")
print(df["competencia"].min(), "->", df["competencia"].max())

print("\n=== Resumo de RBT12 por anexo ===")
resumo = (
    df.groupby("anexo")["rbt12"]
    .agg(["count", "min", "max", "mean"])
    .round(2)
)
display(resumo)

# Fotografia inicial: distribuição do RBT12 por anexo.
fig, ax = plt.subplots()
for anexo, g in df.groupby("anexo"):
    ax.hist(g["rbt12"], bins=20, alpha=0.55, label=f"Anexo {anexo}")
ax.set_title("Distribuição do RBT12 por anexo (estado inicial)")
ax.set_xlabel("RBT12 (R$)")
ax.set_ylabel("Frequência")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1000:,.0f}k"))
ax.legend()
plt.tight_layout()
plt.show()

## 3. Desacoplar RBT12 do Cálculo Automático

O ponto central da sua ideia: o RBT12 deixa de ser **sempre derivado** dos 12 meses
e passa a ter uma coluna **editável** que é a fonte de verdade dos cálculos.

- `rbt12_original` → cópia do RBT12 automático (nunca sobrescrita; serve de histórico).
- `rbt12_editavel` → valor **usado nos cálculos**. Começa igual ao original e pode ser
  sobrescrito por simulação ou entrada manual.
- `rbt12_origem` → indica de onde veio o valor vigente (`automatico`, `manual`, `simulacao`).

In [ ]:
# 3.1 — Preserva o automático e cria a coluna editável como fonte de verdade.
df["rbt12_original"] = df["rbt12"].copy()
df["rbt12_editavel"] = df["rbt12"].copy()
df["rbt12_origem"] = "automatico"

# 3.2 — A partir daqui, TODO cálculo deve ler rbt12_editavel.
#        Para deixar isso explícito, trocamos as referências de cálculo.
COLUNA_RBT12_VIGENTE = "rbt12_editavel"

# Exemplo de função de cálculo (alíquota efetiva) usando a coluna vigente.
FAIXAS_ANEXO_I = [
    (0, 180_000, 0.04, 0),
    (180_000.01, 360_000, 0.073, 5_940),
    (360_000.01, 720_000, 0.095, 13_860),
    (720_000.01, 1_800_000, 0.107, 22_500),
    (1_800_000.01, 3_600_000, 0.143, 87_300),
    (3_600_000.01, 4_800_000, 0.19, 378_000),
]


def obter_faixa(rbt12, faixas=FAIXAS_ANEXO_I):
    for inf, sup, aliq, ded in faixas:
        if inf <= rbt12 <= sup:
            return inf, sup, aliq, ded
    return None


def aliquota_efetiva(rbt12):
    """((RBT12 x aliq nominal) - deducao) / RBT12 — usa o RBT12 vigente."""
    faixa = obter_faixa(rbt12)
    if faixa is None or rbt12 <= 0:
        return 0.0
    _, _, aliq, ded = faixa
    return ((rbt12 * aliq) - ded) / rbt12


# 3.3 — Confirma que o cálculo segue a coluna editável, não mais a crua.
df["aliquota_efetiva"] = df[COLUNA_RBT12_VIGENTE].apply(aliquota_efetiva)

print("Coluna vigente nos cálculos:", COLUNA_RBT12_VIGENTE)
df[["anexo", "competencia", "rbt12_original", "rbt12_editavel",
    "rbt12_origem", "aliquota_efetiva"]].tail(6)

## 4. Simular RBT12 Futuro

Projetamos o RBT12 para meses futuros com três premissas simples:

- **Crescimento percentual** mensal composto (taxa).
- **Sazonalidade** multiplicativa opcional (fatores por mês).
- **Média móvel** como linha de base (últimos 12 meses).

Geramos três cenários — **otimista**, **base** e **pessimista** — armazenados em
colunas separadas para não contaminar o valor editável até que você aplique um deles.

In [ ]:
def projetar_rbt12(rbt12_base, meses_futuros=12, crescimento_mensal=0.01,
                   sazonalidade=None):
    """Projeta o RBT12 futuro combinando crescimento + sazonalidade.

    Parameters
    ----------
    rbt12_base : float
        RBT12 de partida (último valor vigente).
    meses_futuros : int
        Quantidade de meses a projetar.
    crescimento_mensal : float
        Taxa de crescimento mensal composta (ex.: 0.01 = 1% a.m.).
    sazonalidade : list[float] | None
        Fatores multiplicativos por mês (12 posições). None = sem sazonalidade.

    Returns
    -------
    list[float] com a projeção mês a mês.
    """
    projecao = []
    valor = float(rbt12_base)
    for mes in range(meses_futuros):
        fator_saz = 1.0
        if sazonalidade:
            fator_saz = sazonalidade[mes % len(sazonalidade)]
        valor = valor * (1 + crescimento_mensal) * fator_saz
        projecao.append(round(valor, 2))
    return projecao


def simular_cenarios(df, meses_futuros=12,
                     cresc_otimista=0.02, cresc_base=0.008, cresc_pessimista=-0.01,
                     sazonalidade=None):
    """Gera colunas de cenários futuros por anexo, a partir do último RBT12."""
    partes = []
    for anexo, g in df.groupby("anexo"):
        g = g.sort_values("competencia").copy()
        ultimo = g.iloc[-1]
        base = ultimo["rbt12_editavel"]

        futuras = pd.date_range(
            start=ultimo["competencia"] + pd.offsets.MonthBegin(1),
            periods=meses_futuros, freq="MS",
        )
        fut = pd.DataFrame({
            "anexo": anexo,
            "competencia": futuras,
            "rbt12_sim_otimista": projetar_rbt12(base, meses_futuros, cresc_otimista, sazonalidade),
            "rbt12_sim_base": projetar_rbt12(base, meses_futuros, cresc_base, sazonalidade),
            "rbt12_sim_pessimista": projetar_rbt12(base, meses_futuros, cresc_pessimista, sazonalidade),
        })
        partes.append(fut)

    return pd.concat(partes, ignore_index=True)


# Sazonalidade de exemplo (fatores por mês: jan..dez).
SAZONALIDADE = [1.00, 0.95, 1.00, 1.05, 1.10, 1.05, 1.00, 1.00, 1.05, 1.10, 1.15, 1.05]

df_futuro = simular_cenarios(df, meses_futuros=12, sazonalidade=SAZONALIDADE)
print("Projeções geradas:")
display(df_futuro.round(2).head(12))

## 5. Visualizar Histórico de RBT12 por Anexo

Sobrepondo **automático**, **simulado** e **editável** você enxerga:

- o passado (incluindo meses antigos, ex.: 2021, que antes ficavam fora da janela);
- o ponto onde a edição manual divergiu do automático;
- os cenários futuros.

Os filtros por **anexo** e por **período** controlam o recorte.

In [ ]:
def plotar_historico(df, df_futuro, anexo=None,
                     inicio=None, fim=None, mostrar_simulacao=True):
    """Plota a série de RBT12 (automático, editável e cenários) para um anexo."""
    d = df.copy()
    f = df_futuro.copy()
    if anexo:
        d = d[d["anexo"] == anexo]
        f = f[f["anexo"] == anexo]
    if inicio:
        d = d[d["competencia"] >= pd.Timestamp(inicio)]
    if fim:
        d = d[d["competencia"] <= pd.Timestamp(fim)]

    fig, ax = plt.subplots()

    for a, g in d.groupby("anexo"):
        g = g.sort_values("competencia")
        ax.plot(g["competencia"], g["rbt12_original"],
                linestyle="--", alpha=0.6, label=f"Anexo {a} · automático")
        ax.plot(g["competencia"], g["rbt12_editavel"],
                linewidth=2, label=f"Anexo {a} · editável")

        if mostrar_simulacao:
            gg = f[f["anexo"] == a].sort_values("competencia")
            if not gg.empty:
                ax.plot(gg["competencia"], gg["rbt12_sim_base"], ":",
                        label=f"Anexo {a} · sim. base")
                ax.plot(gg["competencia"], gg["rbt12_sim_otimista"], ":", alpha=0.6,
                        label=f"Anexo {a} · sim. otimista")
                ax.plot(gg["competencia"], gg["rbt12_sim_pessimista"], ":", alpha=0.6,
                        label=f"Anexo {a} · sim. pessimista")

    ax.set_title("Histórico e projeção de RBT12 por anexo")
    ax.set_xlabel("Competência")
    ax.set_ylabel("RBT12 (R$)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1000:,.0f}k"))
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()


# Recorte: Anexo I, desde 2021 (para recuperar valores antigos).
plotar_historico(df, df_futuro, anexo="I", inicio="2021-01-01")

# Recorte: todos os anexos, últimos 24 meses + projeção.
plotar_historico(df, df_futuro, inicio="2024-10-01")

## 6. Entrada Manual de RBT12 por Anexo

Aqui mora o "digitar o RBT12 manualmente dentro de cada anexo". A função
`definir_rbt12_manual` recebe `{anexo: valor}` e sobrescreve `rbt12_editavel`
no período vigente, validando:

- o valor é **numérico** e **positivo**;
- respeita o **teto do Simples** (R$ 4.800.000,00) e um piso mínimo;
- registra a **origem** como `manual` (preservando `rbt12_original`).

In [ ]:
TETO_SIMPLES = 4_800_000.00
PISO_RBT12 = 0.00


class RBT12Invalido(ValueError):
    """Erro de validação do RBT12 informado manualmente."""


def validar_rbt12(valor):
    """Valida e normaliza um RBT12 digitado manualmente."""
    if isinstance(valor, str):
        texto = valor.strip().replace(".", "").replace(",", ".")
        if texto == "":
            raise RBT12Invalido("Valor vazio.")
        try:
            valor = float(texto)
        except ValueError as exc:
            raise RBT12Invalido(f"Valor não numérico: {valor!r}") from exc
    if not isinstance(valor, (int, float)) or isinstance(valor, bool):
        raise RBT12Invalido(f"Tipo inválido: {type(valor).__name__}")
    if np.isnan(valor) or np.isinf(valor):
        raise RBT12Invalido("Valor não finito.")
    if valor < PISO_RBT12:
        raise RBT12Invalido(f"RBT12 abaixo do piso ({valor}).")
    if valor > TETO_SIMPLES:
        raise RBT12Invalido(
            f"RBT12 {valor:,.2f} excede o teto do Simples ({TETO_SIMPLES:,.2f})."
        )
    return round(float(valor), 2)


def definir_rbt12_manual(df, valores_por_anexo, competencia=None):
    """Sobrescreve rbt12_editavel para o período vigente, por anexo.

    Parameters
    ----------
    df : DataFrame base (deve ter rbt12_editavel e rbt12_origem).
    valores_por_anexo : dict {anexo: valor}
    competencia : competência-alvo; None = última competência do anexo.

    Returns
    -------
    (df_editado, relatorio) — relatorio traz aplicados e rejeitados.
    """
    df = df.copy()
    relatorio = {"aplicados": [], "rejeitados": []}

    for anexo, valor in valores_por_anexo.items():
        try:
            v = validar_rbt12(valor)
        except RBT12Invalido as exc:
            relatorio["rejeitados"].append({"anexo": anexo, "valor": valor, "erro": str(exc)})
            continue

        mask = df["anexo"] == anexo
        if competencia is not None:
            mask &= df["competencia"] == pd.Timestamp(competencia)
        else:
            # Período vigente = competência mais recente do anexo.
            mask &= df["competencia"] == df.loc[df["anexo"] == anexo, "competencia"].max()

        n = int(mask.sum())
        if n == 0:
            relatorio["rejeitados"].append(
                {"anexo": anexo, "valor": valor, "erro": "Nenhuma linha correspondente."}
            )
            continue

        df.loc[mask, "rbt12_editavel"] = v
        df.loc[mask, "rbt12_origem"] = "manual"
        relatorio["aplicados"].append({"anexo": anexo, "valor": v, "linhas": n})

    # Recalcula a alíquota efetiva com a coluna vigente já atualizada.
    df["aliquota_efetiva"] = df[COLUNA_RBT12_VIGENTE].apply(aliquota_efetiva)
    return df, relatorio


# Exemplo: digita manualmente o RBT12 do Anexo I (aceita vírgula) e testa inválidos.
df, rel = definir_rbt12_manual(df, {"I": "1.250.000,00", "III": 999_999})
print("=== Aplicados ===")
for r in rel["aplicados"]:
    print(" ", r)

# Casos inválidos para demonstrar a validação:
_, rel_erro = definir_rbt12_manual(df, {"I": "abc", "III": 9_000_000})
print("\n=== Rejeitados ===")
for r in rel_erro["rejeitados"]:
    print(" ", r)

Aplicando um cenário simulado sobre o valor editável (ex.: Anexo I, mês mais recente):

```python
# Exemplo: adota o cenário "base" do primeiro mês futuro como RBT12 vigente.
primeiro_futuro = df_futuro.groupby("anexo").first()
valores_sim = primeiro_futuro["rbt12_sim_base"].to_dict()
df, rel_sim = definir_rbt12_manual(df, valores_sim)
# e marca a origem como simulação:
for a in valores_sim:
    mask = (df["anexo"] == a) & (df["competencia"] == df.loc[df["anexo"] == a, "competencia"].max())
    df.loc[mask, "rbt12_origem"] = "simulacao"
```

## 7. Validar Consistência dos Cenários

Compara **original**, **simulado** e **manual** e sinaliza divergências acima de um
*threshold*. O resultado é uma **tabela de auditoria** pronta para exportar (CSV),
útil para revisar quando o RBT12 foi editado à mão e por quê.

In [ ]:
def auditar_consistencia(df, df_futuro, threshold=0.05):
    """Compara original x editável x simulado e sinaliza divergências.

    Retorna uma tabela de auditoria com a diferença percentual e flags.
    """
    base = df.copy()

    # Diferença entre o automático e o valor editável vigente.
    base["dif_pct"] = np.where(
        base["rbt12_original"] > 0,
        (base["rbt12_editavel"] - base["rbt12_original"]) / base["rbt12_original"],
        np.nan,
    )
    base["divergente"] = base["dif_pct"].abs() > threshold

    # Projeção base do mês futuro correspondente (para contexto).
    sim = (
        df_futuro.groupby("anexo")["rbt12_sim_base"]
        .first()
        .rename("rbt12_sim_base")
        .reset_index()
    )
    ult = (
        base.sort_values("competencia")
        .groupby("anexo")
        .tail(1)
        .merge(sim, on="anexo", how="left")
    )
    ult["dif_vs_sim_pct"] = np.where(
        ult["rbt12_editavel"] > 0,
        (ult["rbt12_sim_base"] - ult["rbt12_editavel"]) / ult["rbt12_editavel"],
        np.nan,
    )

    colunas = [
        "anexo", "competencia", "rbt12_original", "rbt12_editavel",
        "rbt12_origem", "dif_pct", "divergente",
        "rbt12_sim_base", "dif_vs_sim_pct",
    ]
    return ult[colunas].round(4)


auditoria = auditar_consistencia(df, df_futuro, threshold=0.05)
display(auditoria)

# Exporta a tabela de auditoria (compatível com Excel/BR: decimal = vírgula).
auditoria.to_csv(
    "auditoria_rbt12.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig"
)
print("Auditoria exportada para auditoria_rbt12.csv")

# Quantos anexos estão divergentes do cálculo automático?
n_div = int(auditoria["divergente"].sum())
print(f"Anexos divergentes (>{5}%): {n_div} de {len(auditoria)}")

## Conclusão

Com o modelo deste notebook, o RBT12 fica **livre**:

| Capacidade | Como |
|---|---|
| **Simular futuro** | `simular_cenarios` gera otimista/base/pessimista a partir do último RBT12 |
| **Ver histórico** | `rbt12_original` preserva o automático; `plotar_historico` plota 2021→futuro |
| **Digitar manualmente** | `definir_rbt12_manual({anexo: valor})` sobrescreve só o vigente, com validação |
| **Auditar** | `auditar_consistencia` sinaliza divergências e exporta CSV |

**Próximo passo (integrar ao app `simples_app`):**
1. Adicionar em `anexos` as colunas `rbt12_manual` (texto vazio = usar automático) e
   `rbt12_origem`, via `_adicionar_coluna` (já existe no `database.py`).
2. Na rota `/anexo/<anexo>`, ter um campo "RBT12 manual" e um toggle
   "usar automático × usar manual"; ao salvar, gravar em `rbt12_manual`.
3. No `calc.calcular`, aceitar o RBT12 manual; se vazio, cair no `calcular_rbt12`.
4. Trocar a janela relativa (últimos 12 meses) por **competência selecionável**, para
   visualizar/editar períodos antigos (2021) sem depender da data atual.